# Thesis checking behavior analysis

This notebook is thesis-specific. It analyzes the expanded-label model predictions
for spatial checking, wall-directed checking/thigmotaxis, and lurching candidates.

It requires frozen zone definitions before full scoring:

- arena boundary
- wall band
- learned-place zone

The notebook will still compute tracking QC and wall/locomotor controls when arena
settings are present, but directed-return endpoints require a learned-place zone.



In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
import math
import pickle
import re

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import numpy as np
import pandas as pd
from IPython.display import display

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass



In [ ]:
ROOT = Path('/content/drive/MyDrive/sp1DLC')
RUN = ROOT / 'Colab_Training_Runs/mobile_worklabels_20260927_210240'
PREDICTIONS = RUN / 'videos'
OUT = RUN / 'Thesis_Checking_Behavior_Analysis' / datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
OUT.mkdir(parents=True, exist_ok=True)

FOCUS = ['nose', 'left_ear', 'right_ear', 'spine_mid', 'left_hip', 'right_hip', 'tail_base']
TORSO = ['spine_mid', 'left_hip', 'right_hip', 'tail_base']

LIKELIHOOD_CUTOFF = 0.20
MAX_INTERPOLATION_GAP_FRAMES = 6
SMOOTHING_WINDOW_FRAMES = 5

# -----------------------
# REQUIRED SCORING INPUTS
# -----------------------
# Set these before using the place-directed checking endpoints.
#
# Circle arena example:
# ARENA = {'kind': 'circle', 'center': (320, 240), 'radius': 210}
#
# Rectangle arena example:
# ARENA = {'kind': 'rect', 'x_min': 40, 'x_max': 600, 'y_min': 30, 'y_max': 450}
#
# Learned-place zone example:
# LEARNED_PLACE = {'kind': 'circle', 'center': (410, 220), 'radius': 45}
#
# If zones vary by video, put entries in VIDEO_ZONES below:
# VIDEO_ZONES = {
#     'Test 30': {
#         'arena': {'kind': 'circle', 'center': (320, 240), 'radius': 210},
#         'learned_place': {'kind': 'circle', 'center': (410, 220), 'radius': 45},
#     },
# }

ARENA = None
LEARNED_PLACE = None
VIDEO_ZONES = {}

WALL_BAND_PX = 35
DEPARTURE_MIN_FRAMES = 6
RETURN_MIN_FRAMES = 3
EVENT_MERGE_GAP_FRAMES = 12
MIN_BOUT_FRAMES = 6
LURCH_QUANTILE = 0.98

plt.rcParams.update({
    'figure.dpi': 180,
    'savefig.dpi': 180,
    'font.size': 10,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.18,
    'axes.facecolor': '#fbfbf8',
    'figure.facecolor': 'white',
})

print('Prediction folder:', PREDICTIONS)
print('Output folder:', OUT)



In [ ]:
def natural_key(path: Path) -> tuple:
    return tuple(int(part) if part.isdigit() else part.lower() for part in re.split(r'(\d+)', path.name))


def video_name(path: Path) -> str:
    return path.name.partition('DLC_')[0]


def h5_files() -> list[Path]:
    files = sorted(PREDICTIONS.glob('*.h5'), key=natural_key)
    files = [path for path in files if 'DLC_' in path.name or 'snapshot' in path.name]
    assert files, f'No prediction H5 files found in {PREDICTIONS}. Run the worklabels video notebook first.'
    return files


def read_prediction(path: Path) -> pd.DataFrame:
    df = pd.read_hdf(path)
    assert isinstance(df.columns, pd.MultiIndex), f'{path.name} does not have DLC MultiIndex columns'
    missing = [part for part in FOCUS if part not in df.columns.get_level_values('bodyparts')]
    assert not missing, f'{path.name} is missing bodyparts: {missing}'
    return df.loc[:, df.columns.get_level_values('bodyparts').isin(FOCUS)]


def read_fps(path: Path) -> float | None:
    meta_path = path.with_name(path.stem + '_meta.pickle')
    if not meta_path.exists():
        return None
    try:
        with meta_path.open('rb') as handle:
            meta = pickle.load(handle)
    except Exception:
        return None
    stack = [meta]
    while stack:
        item = stack.pop()
        if isinstance(item, dict):
            for key, value in item.items():
                if str(key).lower() in {'fps', 'framerate', 'frame_rate'}:
                    try:
                        fps = float(value)
                        if math.isfinite(fps) and fps > 0:
                            return fps
                    except Exception:
                        pass
                elif isinstance(value, (dict, list, tuple)):
                    stack.append(value)
        elif isinstance(item, (list, tuple)):
            stack.extend(item)
    return None


def get_coord(df: pd.DataFrame, bodypart: str, coord: str) -> pd.Series:
    cols = [
        col for col in df.columns
        if col[df.columns.names.index('bodyparts')] == bodypart
        and col[df.columns.names.index('coords')] == coord
    ]
    assert len(cols) == 1, f'Expected exactly one {bodypart}/{coord} column'
    return pd.to_numeric(df[cols[0]], errors='coerce')


def clean_series(series: pd.Series) -> pd.Series:
    filled = series.interpolate(limit=MAX_INTERPOLATION_GAP_FRAMES, limit_direction='both')
    if SMOOTHING_WINDOW_FRAMES > 1:
        filled = filled.rolling(SMOOTHING_WINDOW_FRAMES, center=True, min_periods=1).median()
    return filled


def point_table(df: pd.DataFrame) -> dict[str, pd.DataFrame]:
    out = {}
    for part in FOCUS:
        likelihood = get_coord(df, part, 'likelihood')
        valid = likelihood >= LIKELIHOOD_CUTOFF
        out[part] = pd.DataFrame({
            'x': clean_series(get_coord(df, part, 'x').where(valid)),
            'y': clean_series(get_coord(df, part, 'y').where(valid)),
            'likelihood': likelihood,
            'valid': valid,
        })
    return out


def body_center(points: dict[str, pd.DataFrame]) -> pd.DataFrame:
    x = pd.concat([points[part]['x'].rename(part) for part in TORSO], axis=1).mean(axis=1, skipna=True)
    y = pd.concat([points[part]['y'].rename(part) for part in TORSO], axis=1).mean(axis=1, skipna=True)
    valid_count = pd.concat([points[part]['valid'].rename(part) for part in TORSO], axis=1).sum(axis=1)
    center = pd.DataFrame({'x': clean_series(x), 'y': clean_series(y), 'valid_torso_points': valid_count})
    center['usable'] = center[['x', 'y']].notna().all(axis=1)
    center['step_px'] = np.hypot(center['x'].diff(), center['y'].diff()).where(center['usable'])
    center['accel_px'] = center['step_px'].diff().abs()
    return center


def in_zone(x: pd.Series, y: pd.Series, zone: dict | None) -> pd.Series:
    if zone is None:
        return pd.Series(False, index=x.index)
    if zone['kind'] == 'circle':
        cx, cy = zone['center']
        return np.hypot(x - cx, y - cy) <= zone['radius']
    if zone['kind'] == 'rect':
        return (x >= zone['x_min']) & (x <= zone['x_max']) & (y >= zone['y_min']) & (y <= zone['y_max'])
    raise ValueError(f'Unknown zone kind: {zone}')


def wall_zone(x: pd.Series, y: pd.Series, arena: dict | None, band_px: float = WALL_BAND_PX) -> pd.Series:
    if arena is None:
        return pd.Series(False, index=x.index)
    if arena['kind'] == 'circle':
        cx, cy = arena['center']
        dist = np.hypot(x - cx, y - cy)
        return (arena['radius'] - dist) <= band_px
    if arena['kind'] == 'rect':
        distances = pd.concat([
            (x - arena['x_min']).abs(),
            (arena['x_max'] - x).abs(),
            (y - arena['y_min']).abs(),
            (arena['y_max'] - y).abs(),
        ], axis=1)
        return distances.min(axis=1) <= band_px
    raise ValueError(f'Unknown arena kind: {arena}')


def bouts(mask: pd.Series, min_frames: int = MIN_BOUT_FRAMES, merge_gap: int = EVENT_MERGE_GAP_FRAMES) -> list[tuple[int, int]]:
    values = mask.fillna(False).astype(bool).to_numpy()
    raw = []
    start = None
    for i, value in enumerate(values):
        if value and start is None:
            start = i
        if (not value or i == len(values) - 1) and start is not None:
            end = i if value and i == len(values) - 1 else i - 1
            if end - start + 1 >= min_frames:
                raw.append((start, end))
            start = None
    if not raw:
        return []
    merged = [raw[0]]
    for start, end in raw[1:]:
        prev_start, prev_end = merged[-1]
        if start - prev_end <= merge_gap:
            merged[-1] = (prev_start, end)
        else:
            merged.append((start, end))
    return merged


def first_true_run_end(mask: pd.Series, min_frames: int) -> int | None:
    values = mask.fillna(False).astype(bool).to_numpy()
    run = 0
    for i, value in enumerate(values):
        run = run + 1 if value else 0
        if run >= min_frames:
            return i
    return None


def path_length(center: pd.DataFrame, start: int, end: int) -> float:
    return float(center.loc[start:end, 'step_px'].sum(skipna=True))


def straight_distance(center: pd.DataFrame, start: int, end: int) -> float:
    x0, y0 = center.loc[start, ['x', 'y']]
    x1, y1 = center.loc[end, ['x', 'y']]
    return float(np.hypot(x1 - x0, y1 - y0))


def direction_label(center: pd.DataFrame, frame: int, arena: dict | None, place: dict | None) -> str:
    x, y = center.loc[frame, ['x', 'y']]
    if place and bool(in_zone(pd.Series([x]), pd.Series([y]), place).iloc[0]):
        return 'place'
    if arena and bool(wall_zone(pd.Series([x]), pd.Series([y]), arena).iloc[0]):
        return 'wall'
    return 'other'



In [ ]:
raw = {}
for path in h5_files():
    name = video_name(path)
    df = read_prediction(path)
    points = point_table(df)
    center = body_center(points)
    zones = VIDEO_ZONES.get(name, {})
    arena = zones.get('arena', ARENA)
    place = zones.get('learned_place', LEARNED_PLACE)
    raw[name] = {
        'path': path,
        'df': df,
        'points': points,
        'center': center,
        'fps': read_fps(path),
        'arena': arena,
        'place': place,
    }

all_accel = pd.concat([item['center']['accel_px'] for item in raw.values()], ignore_index=True).dropna()
lurch_threshold = float(all_accel.quantile(LURCH_QUANTILE)) if len(all_accel) else np.nan

print('Videos:', len(raw))
print('Lurch candidate acceleration threshold:', round(lurch_threshold, 3) if math.isfinite(lurch_threshold) else 'not available')
if LEARNED_PLACE is None and not any('learned_place' in zones for zones in VIDEO_ZONES.values()):
    print('NOTE: learned-place zone is not set. Place-directed return endpoints will be blank until zones are defined.')



In [ ]:
summary_rows = []
event_rows = []
lurch_rows = []
qc_rows = []

for name, item in raw.items():
    center = item['center']
    fps = item['fps'] or np.nan
    seconds = (lambda frames: frames / fps) if math.isfinite(fps) else (lambda frames: np.nan)
    arena = item['arena']
    place = item['place']
    in_place = in_zone(center['x'], center['y'], place)
    near_wall = wall_zone(center['x'], center['y'], arena)
    outside_place = ~in_place

    first_departure = first_true_run_end(outside_place, DEPARTURE_MIN_FRAMES) if place else None
    post_departure = pd.Series(False, index=center.index)
    if first_departure is not None:
        post_departure.loc[first_departure:] = True
    return_bouts = bouts(in_place & post_departure, min_frames=RETURN_MIN_FRAMES)
    wall_bouts = bouts(near_wall, min_frames=MIN_BOUT_FRAMES)

    return_latencies = []
    efficiencies = []
    for idx, (start, end) in enumerate(return_bouts, start=1):
        latency_frames = start - first_departure if first_departure is not None else np.nan
        return_latencies.append(seconds(latency_frames) if math.isfinite(fps) else latency_frames)
        direct = straight_distance(center, first_departure, start) if first_departure is not None else np.nan
        observed = path_length(center, first_departure, start) if first_departure is not None else np.nan
        efficiency = direct / observed if observed and observed > 0 else np.nan
        efficiencies.append(efficiency)
        event_rows.append({
            'video': name,
            'event_type': 'place_return',
            'event_index': idx,
            'start_frame': start,
            'end_frame': end,
            'duration_s': seconds(end - start + 1),
            'latency_from_first_departure_s_or_frames': return_latencies[-1],
            'path_efficiency': efficiency,
        })

    for idx, (start, end) in enumerate(wall_bouts, start=1):
        event_rows.append({
            'video': name,
            'event_type': 'wall_bout',
            'event_index': idx,
            'start_frame': start,
            'end_frame': end,
            'duration_s': seconds(end - start + 1),
            'latency_from_first_departure_s_or_frames': np.nan,
            'path_efficiency': np.nan,
        })

    lurch_mask = center['accel_px'] >= lurch_threshold if math.isfinite(lurch_threshold) else pd.Series(False, index=center.index)
    for idx, (start, end) in enumerate(bouts(lurch_mask, min_frames=1, merge_gap=3), start=1):
        peak = int(center.loc[start:end, 'accel_px'].idxmax())
        label = direction_label(center, peak, arena, place)
        lurch_rows.append({
            'video': name,
            'event_index': idx,
            'start_frame': start,
            'end_frame': end,
            'peak_frame': peak,
            'duration_s': seconds(end - start + 1),
            'peak_accel_px': float(center.loc[peak, 'accel_px']),
            'direction_candidate': label,
        })

    summary_rows.append({
        'video': name,
        'fps_from_metadata': item['fps'],
        'frames': len(center),
        'usable_center_pct': float(center['usable'].mean() * 100),
        'total_distance_px': float(center['step_px'].sum(skipna=True)),
        'median_step_px_per_frame': float(center['step_px'].median(skipna=True)),
        'place_zone_defined': place is not None,
        'arena_defined': arena is not None,
        'directed_returns_after_first_departure': len(return_bouts) if place else np.nan,
        'latency_to_first_departure_s': seconds(first_departure) if first_departure is not None else np.nan,
        'place_dwell_s': seconds(int(in_place.sum())) if place else np.nan,
        'place_dwell_pct': float(in_place.mean() * 100) if place else np.nan,
        'wall_dwell_s': seconds(int(near_wall.sum())) if arena else np.nan,
        'wall_dwell_pct': float(near_wall.mean() * 100) if arena else np.nan,
        'wall_checking_bout_count': len(wall_bouts) if arena else np.nan,
        'first_return_latency_s_or_frames': return_latencies[0] if return_latencies else np.nan,
        'mean_return_latency_s_or_frames': float(np.nanmean(return_latencies)) if return_latencies else np.nan,
        'mean_return_path_efficiency': float(np.nanmean(efficiencies)) if efficiencies else np.nan,
        'lurch_candidate_count': int(sum(1 for row in lurch_rows if row['video'] == name)),
        'lurch_candidate_place_directed': int(sum(1 for row in lurch_rows if row['video'] == name and row['direction_candidate'] == 'place')),
        'lurch_candidate_wall_directed': int(sum(1 for row in lurch_rows if row['video'] == name and row['direction_candidate'] == 'wall')),
    })

    for part in FOCUS:
        point = item['points'][part]
        qc_rows.append({
            'video': name,
            'bodypart': part,
            'visible_pct': float(point['valid'].mean() * 100),
            'median_likelihood': float(point['likelihood'].median(skipna=True)),
        })

summary_df = pd.DataFrame(summary_rows).sort_values('video', key=lambda s: s.map(lambda value: natural_key(Path(str(value)))))
events_df = pd.DataFrame(event_rows)
lurch_df = pd.DataFrame(lurch_rows)
qc_df = pd.DataFrame(qc_rows)

summary_df.to_csv(OUT / 'thesis_checking_summary.csv', index=False)
events_df.to_csv(OUT / 'thesis_checking_events.csv', index=False)
lurch_df.to_csv(OUT / 'lurch_candidate_events.csv', index=False)
qc_df.to_csv(OUT / 'tracking_bodypart_qc.csv', index=False)

display(summary_df.round(2))
display(events_df.head(20).round(2))
display(lurch_df.head(20).round(2))



In [ ]:
def save_bar(values: pd.Series, title: str, xlabel: str, filename: str, color: str) -> None:
    fig_h = max(5.5, 0.22 * len(values) + 2.0)
    fig, ax = plt.subplots(figsize=(9.5, fig_h))
    ordered = values.sort_values()
    ax.barh(ordered.index, ordered.values, color=color, alpha=0.9)
    ax.set_title(title, loc='left', fontsize=14, weight='bold')
    ax.set_xlabel(xlabel)
    ax.set_ylabel('')
    fig.tight_layout()
    fig.savefig(OUT / filename, bbox_inches='tight')
    plt.show()


save_bar(summary_df.set_index('video')['total_distance_px'], 'Locomotor control', 'Total body-center distance, pixels', 'control_total_distance.png', '#6f5d2f')
save_bar(summary_df.set_index('video')['usable_center_pct'], 'Tracking QC', '% usable body-center frames', 'tracking_coverage.png', '#2f6f73')

if summary_df['place_zone_defined'].any():
    save_bar(summary_df.set_index('video')['directed_returns_after_first_departure'].dropna(), 'Place-directed checking', 'Directed returns after first departure', 'place_directed_returns.png', '#654b7c')
    save_bar(summary_df.set_index('video')['place_dwell_pct'].dropna(), 'Place dwell', '% frames in learned-place zone', 'place_dwell_pct.png', '#84693d')

if summary_df['arena_defined'].any():
    save_bar(summary_df.set_index('video')['wall_dwell_pct'].dropna(), 'Wall-directed behavior / thigmotaxis', '% frames in wall band', 'wall_dwell_pct.png', '#7b4b3a')
    save_bar(summary_df.set_index('video')['wall_checking_bout_count'].dropna(), 'Wall-directed bouts', 'Wall-band bout count', 'wall_bouts.png', '#2f5f84')

save_bar(summary_df.set_index('video')['lurch_candidate_count'], 'Lurching candidates', 'High-acceleration candidate count', 'lurch_candidate_count.png', '#8a4f7d')



In [ ]:
def draw_zone(ax, zone: dict | None, color: str, label: str) -> None:
    if not zone:
        return
    if zone['kind'] == 'circle':
        circle = plt.Circle(zone['center'], zone['radius'], fill=False, color=color, lw=1.8, label=label)
        ax.add_patch(circle)
    elif zone['kind'] == 'rect':
        rect = plt.Rectangle((zone['x_min'], zone['y_min']), zone['x_max'] - zone['x_min'], zone['y_max'] - zone['y_min'], fill=False, color=color, lw=1.8, label=label)
        ax.add_patch(rect)


def plot_video_page(name: str, item: dict, pdf: PdfPages | None = None) -> None:
    center = item['center']
    place = item['place']
    arena = item['arena']
    ok = center['usable']
    fig, axes = plt.subplots(1, 2, figsize=(12, 5.2))

    ax = axes[0]
    if ok.any():
        ax.plot(center.loc[ok, 'x'], center.loc[ok, 'y'], color='#2f6f73', lw=1.0)
    draw_zone(ax, arena, '#7b4b3a', 'arena')
    draw_zone(ax, place, '#654b7c', 'learned place')
    ax.invert_yaxis()
    ax.set_aspect('equal', adjustable='box')
    ax.set_title(f'{name}: trajectory with zones', loc='left', weight='bold')
    ax.set_xlabel('x pixel')
    ax.set_ylabel('y pixel')
    ax.legend(frameon=False, fontsize=8)

    ax = axes[1]
    ax.plot(center.index, center['step_px'], color='#6f5d2f', lw=0.8, label='step')
    ax2 = ax.twinx()
    ax2.plot(center.index, center['accel_px'], color='#8a4f7d', lw=0.6, alpha=0.65, label='accel')
    if math.isfinite(lurch_threshold):
        ax2.axhline(lurch_threshold, color='#8a4f7d', ls='--', lw=1)
    ax.set_title('Movement and lurch-candidate signal', loc='left', weight='bold')
    ax.set_xlabel('Frame')
    ax.set_ylabel('Pixels/frame')
    ax2.set_ylabel('Acceleration px/frame')

    fig.tight_layout()
    if pdf:
        pdf.savefig(fig, bbox_inches='tight')
    plt.show()


pdf_path = OUT / 'thesis_checking_video_pages.pdf'
with PdfPages(pdf_path) as pdf:
    for name in summary_df['video']:
        plot_video_page(name, raw[name], pdf=pdf)
print('Saved:', pdf_path)



In [ ]:
report = OUT / 'thesis_checking_report.md'
with report.open('w', encoding='utf-8') as handle:
    handle.write('# Thesis checking behavior report\n\n')
    handle.write(f'Generated: {datetime.now(timezone.utc).isoformat()}\n\n')
    handle.write(f'Run folder: `{RUN}`\n\n')
    handle.write('## Scope\n\n')
    handle.write('This analysis is aligned with the thesis scope: place-directed checking, wall-directed checking/thigmotaxis, and lurching candidates. Place-directed endpoints require frozen learned-place zones. Wall-directed endpoints require frozen arena boundaries. Lurching remains exploratory until manually validated.\n\n')
    handle.write('## Endpoint hierarchy\n\n')
    handle.write('- Primary: directed returns after first departure.\n')
    handle.write('- Secondary: latency to first departure, return latency, place dwell, return path efficiency, wall dwell/thigmotaxis.\n')
    handle.write('- Exploratory: lurching candidates and micro-behavioral bout structure.\n\n')
    handle.write('## Current zone status\n\n')
    handle.write(f'- Videos with learned-place zone defined: {int(summary_df[\"place_zone_defined\"].sum())} of {len(summary_df)}\n')
    handle.write(f'- Videos with arena boundary defined: {int(summary_df[\"arena_defined\"].sum())} of {len(summary_df)}\n\n')
    handle.write('## Files\n\n')
    for name in [
        'thesis_checking_summary.csv',
        'thesis_checking_events.csv',
        'lurch_candidate_events.csv',
        'tracking_bodypart_qc.csv',
        'thesis_checking_video_pages.pdf',
    ]:
        handle.write(f'- `{name}`\n')

print('Saved:', report)
print('Analysis complete. Output folder:', OUT)
